# 02. Handle missing data

This notebook summarizes missingness, removes variables that were not collected
consistently, and creates the complete-case dataset used by the primary analysis.

## Main tasks

- Report missingness for every candidate predictor retained for modelling.
- Document why `internet` and `family_worry` are excluded as variables.
- Use complete-case eligibility across the 19 candidate predictors and `body_image`,
  preserving the original analytic-cohort rule. `body_image` is retained only as a
  cohort-completeness field and is not entered into any model.
- Perform no imputation.
- Update the participant flow and save the processed dataset.

The output count is determined by the data after these rules are applied. The
`body_image` completeness rule is retained to preserve the original analytic cohort.


In [1]:
# Imports and paths
import os
import pandas as pd
from pathlib import Path

# Use environment-based paths before publication; local paths are retained here.
BASE = Path(os.environ.get("METS_DATA", r"D:\eda\mets\data"))
OUT  = Path(os.environ.get("METS_OUT",  r"D:\eda\mets\output"))
for _d in (BASE, BASE/"raw", BASE/"work", OUT, OUT/"tables", OUT/"figures", OUT/"logs"):
    _d.mkdir(parents=True, exist_ok=True)

# Outcomes from notebook 01: IDF primary and modified ATP III sensitivity.
LABELS = ["MS", "MS_atp3"]

# This notebook records its own segment of the participant flow. Notebook 04
# joins the segments in order.
FLOW = []

def keep(d, mask, stage):
    """Keep selected rows and record exclusions, including positive cases."""
    mask = mask.reindex(d.index).fillna(False).astype(bool)
    n_before = len(d)
    out = d.loc[mask].copy()
    excluded = d.loc[~mask]

    record = {"step": len(FLOW), "stage": stage,
              "excluded": n_before - len(out), "remaining": len(out)}
    for label in LABELS:
        if label in d.columns:
            record[f"excluded_{label}"] = int(excluded[label].eq(1).sum())
    FLOW.append(record)

    print(f"  - {stage:<53} -{n_before-len(out):>8,}  ->  {len(out):>9,}"
          f"   (MetS {record.get('excluded_MS', 0):>4,})")
    return out

def save_flow():
    """Write this notebook's flow segment, which notebook 04 draws as Fig 1."""
    flow = pd.DataFrame(FLOW)
    flow.to_csv(OUT/"tables"/"flow_02.csv", index=False, encoding="utf-8-sig")
    return flow


In [2]:
df = pd.read_csv(BASE/"work"/"Obesity_Data_proprecessing.csv")
print(f"loaded {len(df):,} rows, {df.shape[1]} cols")
for label in LABELS:
    print(f"  {label:<12} positive {int(df[label].eq(1).sum()):>6,}"
          f"  ({df[label].mean()*100:5.2f}%)")


loaded 13,984 rows, 47 cols
  MS           positive  1,787  (12.78%)
  MS_atp3      positive  2,271  (16.24%)


## 1. Table 4: missingness among all candidate predictors

In [3]:
# Table 4: missingness among all candidate predictors retained for modelling.
MODEL_FEATURES = [
    "sex", "age", "global_grade", "bmi",
    "chol", "LDL", "AST", "ALT",
    "sleep", "PA", "EatingIndex",
    "weight_control_diet", "weight_control_exercise",
    "weight_control_medication", "weight_control_attempt",
    "bully", "runaway", "want_counseling", "request_counseling",
]


# `body_image` is not a model predictor, but it remains part of the original
# complete-case cohort definition so that the analytic sample is preserved.
COHORT_COMPLETENESS_FIELDS = MODEL_FEATURES + (["body_image"] if "body_image" in df.columns else [])

GROUP = {
    "sex": "Sociodemographic", "age": "Sociodemographic",
    "global_grade": "Sociodemographic", "bmi": "Anthropometric",
    "chol": "Biochemical", "LDL": "Biochemical", "AST": "Biochemical", "ALT": "Biochemical",
    "sleep": "Lifestyle / psychosocial", "PA": "Lifestyle / psychosocial",
    "EatingIndex": "Lifestyle / psychosocial",
    "weight_control_diet": "Weight control",
    "weight_control_exercise": "Weight control",
    "weight_control_medication": "Weight control",
    "weight_control_attempt": "Weight control",
    "bully": "Lifestyle / psychosocial", "runaway": "Lifestyle / psychosocial",
    "want_counseling": "Lifestyle / psychosocial",
    "request_counseling": "Lifestyle / psychosocial",
}

missing = [c for c in MODEL_FEATURES if c not in df.columns]
assert not missing, f"candidate predictor(s) absent: {missing}"

missing_retained = pd.DataFrame({
    "variable": MODEL_FEATURES,
    "group": [GROUP[c] for c in MODEL_FEATURES],
    "n_missing": [int(df[c].isna().sum()) for c in MODEL_FEATURES],
})
missing_retained["missing_%"] = (
    missing_retained["n_missing"] / len(df) * 100
).round(3)
missing_retained["n_complete"] = len(df) - missing_retained["n_missing"]

if "body_image" in df.columns:
    body_row = pd.DataFrame([{
        "variable": "body_image",
        "group": "Cohort completeness (not model predictor)",
        "n_missing": int(df["body_image"].isna().sum()),
        "missing_%": round(float(df["body_image"].isna().mean()*100), 3),
        "n_complete": int(df["body_image"].notna().sum()),
    }])
    missing_retained = pd.concat([missing_retained, body_row], ignore_index=True)

print(missing_retained.to_string(index=False))
n_complete = int(df[COHORT_COMPLETENESS_FIELDS].notna().all(axis=1).sum())
print(f"\ncomplete on all {len(MODEL_FEATURES)} candidate predictors plus body_image: "
      f"{n_complete:,} / {len(df):,}")
print(f"highest missingness among cohort-completeness fields: "
      f"{missing_retained['missing_%'].max():.2f}%")
print("No values are imputed; the primary analysis uses complete cases on the")
print("19 candidate predictors plus body_image. body_image is not a model predictor.")

missing_retained.to_csv(
    OUT/"tables"/"Table4_missing_data.csv",
    index=False, encoding="utf-8-sig"
)



                 variable                                     group  n_missing  missing_%  n_complete
                      sex                          Sociodemographic          0      0.000       13984
                      age                          Sociodemographic          0      0.000       13984
             global_grade                          Sociodemographic          0      0.000       13984
                      bmi                            Anthropometric          0      0.000       13984
                     chol                               Biochemical          3      0.021       13981
                      LDL                               Biochemical         26      0.186       13958
                      AST                               Biochemical          3      0.021       13981
                      ALT                               Biochemical          5      0.036       13979
                    sleep                  Lifestyle / psychosocial         66    

## 2. Missingness in the two variables that are dropped


In [4]:
# Missingness patterns for the variables excluded from the analysis.
GRADE_LABEL = {1: "Elementary", 2: "Middle", 3: "High"}

# Stop if either expected variable is absent.
EXCLUDED_VARS = ["internet", "family_worry"]
missing = [v for v in EXCLUDED_VARS if v not in df.columns]
assert not missing, f"column(s) built in notebook 01 are absent: {missing}"

rows = []
for variable in EXCLUDED_VARS:
    table = (df.assign(_missing=df[variable].isna())
               .groupby(["year", "global_grade"], dropna=False)
               .agg(n=("_missing", "size"), n_missing=("_missing", "sum"))
               .reset_index())
    table["variable"] = variable
    table["grade"] = table["global_grade"].map(GRADE_LABEL)
    table["missing_%"] = (table.n_missing / table.n * 100).round(1)
    rows.append(table[["variable", "year", "grade", "n", "n_missing",
                       "missing_%"]])

missing_excluded = pd.concat(rows, ignore_index=True)
print(missing_excluded.to_string(index=False))
missing_excluded.to_csv(OUT/"tables"/"missing_data_excluded_records.csv",
          index=False, encoding="utf-8-sig")


    variable  year      grade    n  n_missing  missing_%
    internet  2021 Elementary  432          0        0.0
    internet  2021     Middle 1567          0        0.0
    internet  2021       High 1704          0        0.0
    internet  2022 Elementary  413          0        0.0
    internet  2022     Middle 1386          3        0.2
    internet  2022       High 1498          0        0.0
    internet  2023 Elementary  362          0        0.0
    internet  2023     Middle 1318          0        0.0
    internet  2023       High 1857          1        0.1
    internet  2024 Elementary  379         40       10.6
    internet  2024     Middle 1213       1177       97.0
    internet  2024       High 1855          2        0.1
family_worry  2021 Elementary  432        432      100.0
family_worry  2021     Middle 1567          0        0.0
family_worry  2021       High 1704          0        0.0
family_worry  2022 Elementary  413        413      100.0
family_worry  2022     Middle 1

## 3. Remove records incomplete on candidate predictors or body image


In [5]:
print("=== Participant flow (continued) ===")
df = keep(
    df,
    df[COHORT_COMPLETENESS_FIELDS].notna().all(axis=1),
    "Missing candidate predictor or body_image"
)


=== Participant flow (continued) ===
  - Missing candidate predictor or body_image             -     153  ->     13,831   (MetS   24)


In [6]:
# Remove inconsistently collected variables without excluding participants.
n_cols_before = df.shape[1]
df_final = df.drop(columns=EXCLUDED_VARS)
print(f"dropped 2 variables: {n_cols_before} -> {df_final.shape[1]} columns")

for label in LABELS:
    print(f"  {label:<12} positive {int(df_final[label].eq(1).sum()):>6,}"
          f"  ({df_final[label].mean()*100:5.2f}%)")

df_final.to_csv(BASE/"work"/"Obesity_Data_proprecessing_Na.csv",
                index=False, encoding="utf-8-sig")
flow = save_flow()
print(f"\nsaved -> Obesity_Data_proprecessing_Na.csv  ({len(df_final):,} rows)")
print(flow.to_string(index=False))


dropped 2 variables: 47 -> 45 columns
  MS           positive  1,763  (12.75%)
  MS_atp3      positive  2,240  (16.20%)

saved -> Obesity_Data_proprecessing_Na.csv  (13,831 rows)
 step                                     stage  excluded  remaining  excluded_MS  excluded_MS_atp3
    0 Missing candidate predictor or body_image       153      13831           24                31


---

## Next

Notebook 03 reads `Obesity_Data_proprecessing_Na.csv`, applies the predefined
data-quality range rules, and creates the final analytic sample.
